# Lab 5 — sklearn Pipeline

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Combine **preprocessing** and **classification** in one `Pipeline`.
2. Use `ColumnTransformer` to apply different transforms to numeric vs categorical columns.
3. Fit and evaluate on the same train/test split as earlier labs (800 / 200).
4. Explain why preprocessing must live **inside** the pipeline (no data leakage).

> **Checkpoints:** steps `['preprocess', 'clf']` · test accuracy ≈ **0.64** · train 800 / test 200



## Why a Pipeline?

Labs 2–4 scaled or encoded features **manually** before fitting. That works in notebooks but breaks in production:

| Problem | Without Pipeline | With Pipeline |
|---------|------------------|---------------|
| **Data leakage** | Fit scaler on full data, then split | Fit scaler on **train only** inside `pipe.fit()` |
| **Deployment** | Remember transform order by hand | `pipe.predict(new_row)` applies everything |
| **Cross-validation** | Easy to leak labels across folds | `cross_val_score(pipe, X, y)` is safe |

```text
  Raw DataFrame  →  ColumnTransformer  →  LogisticRegression  →  predictions
                     (scale + one-hot)
```


---

## 1. Load data and define feature groups

Numeric columns are **scaled** (zero mean, unit variance). Categorical columns are **one-hot encoded**.

These match `_data` helpers (instructor):

| Type | Columns |
|------|---------|
| Numeric | `loan_amnt`, `int_rate`, `annual_inc`, `dti`, `installment` |
| Categorical | `grade`, `term` |


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
CATEGORICAL_FEATURES = ["grade", "term"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df[feature_cols]
y = df["default"]

print(f"shape: {X.shape}")
print(f"default rate: {y.mean():.3f}")
display(X.head(3))


---

## 2. Train/test split (same as Labs 2–4)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}, test size: {len(X_test)}")
print(f"train default rate: {y_train.mean():.3f}")
print(f"test default rate: {y_test.mean():.3f}")


---

## 3. Build `ColumnTransformer`

Each branch runs only on its column subset. `handle_unknown='ignore'` prevents errors when a new category appears at scoring time.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

preprocessor


---

## 4. Wrap in `Pipeline` and fit


In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipe.fit(X_train, y_train)
print(f"pipeline steps: {[name for name, _ in pipe.steps]}")


---

## 5. Predict and measure accuracy

Adding categoricals (`grade`, `term`) typically lifts accuracy slightly over the numeric-only model from Lab 3 (~0.59).


In [ ]:
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"test accuracy: {accuracy:.4f}")
print(f"sample predictions (first 5): {y_pred[:5].tolist()}")


---

## 6. Peek inside: transformed feature count

After one-hot encoding, the design matrix has more columns than the raw seven features.


In [ ]:
X_train_transformed = pipe.named_steps["preprocess"].transform(X_train)
print(f"transformed train shape: {X_train_transformed.shape}")

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
all_names = list(NUMERIC_FEATURES) + list(cat_names)
print(f"feature count after encoding: {len(all_names)}")
print("sample encoded names:", all_names[:8], "...")


---

## 7. Extension — numeric-only baseline

Compare to a pipeline that uses **only** numeric features (like Lab 3). You should see lower accuracy without grade/term.


In [ ]:
pipe_numeric = Pipeline(
    steps=[
        ("preprocess", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipe_numeric.fit(X_train[NUMERIC_FEATURES], y_train)
acc_numeric = accuracy_score(y_test, pipe_numeric.predict(X_test[NUMERIC_FEATURES]))

print(f"numeric-only accuracy: {acc_numeric:.4f}")
print(f"full pipeline accuracy: {accuracy:.4f}")
print(f"lift from categoricals: {accuracy - acc_numeric:+.4f}")


---

## 8. Checkpoint summary


In [ ]:
step_names = [name for name, _ in pipe.steps]
assert step_names == ["preprocess", "clf"]
assert len(X_train) == 800 and len(X_test) == 200
assert abs(accuracy - 0.6350) < 0.02
print("✓ All checkpoint assertions passed")


## Feature selection (course topic)

<!-- cisco-topic-coverage -->

Feature **engineering** creates columns; feature **selection** keeps the most predictive subset.

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split

num_cols = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
cat_cols = ["grade", "term"]
X_all = df[num_cols + cat_cols]
y_all = df["default"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# score numeric features only (quick classroom demo)
selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X_tr[num_cols], y_tr)
scores = pd.DataFrame({"feature": num_cols, "score": selector.scores_}).sort_values("score", ascending=False)
print("top numeric features by ANOVA F-score:")
print(scores.round(2).to_string(index=False))


---

## Reflection questions

1. What would go wrong if you called `StandardScaler().fit(X)` on the full dataset before `train_test_split`?
2. Why is `handle_unknown='ignore'` useful when deploying to production?
3. How would you add a `GridSearchCV` step to tune `C` in logistic regression without leaking test data?

**Previous:** [Lab 4 — ROC and AUC](lab04_roc_auc.ipynb)  
**Next:** [Lab 6 — SHAP interpretability](lab06_shap_interpretability.ipynb)
